# Grad-CAM & Saliency (중요 픽셀 시각화) — Digit Recognizer (PyTorch) — Colab

학습된 CNN이 **이미지의 어느 픽셀을 보고 판단했는지**를 Grad-CAM과 saliency 맵으로 시각화하는 기본 예제입니다.

- 핵심 흐름: CNN 학습 → **Grad-CAM**(마지막 conv 활성×그래디언트) + **Saliency**(입력 그래디언트) → 중요 픽셀 히트맵.
- IOAI 2025 **Pixel Efficiency** 과제(사진에서 가장 중요한 픽셀 식별)의 기반 기법입니다.
  (공식 해법은 CLIP 비전 임베딩 기여도를 사용 — 여기서는 가장 기본인 Grad-CAM/Saliency 로 연습)
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 데이터를 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비 & CNN 학습

Grad-CAM 을 걸 **마지막 conv 층**을 별도 속성으로 둔 CNN을 학습합니다.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
X = df.drop("label", axis=1).values.reshape(-1,1,28,28).astype("float32")/255.0
y = df["label"].values.astype("int64")
rng = np.random.RandomState(42); idx = rng.permutation(len(X)); cut=int(len(X)*0.9)
tr, va = idx[:cut], idx[cut:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(TensorDataset(torch.from_numpy(X[tr]), torch.from_numpy(y[tr])), batch_size=128, shuffle=True)
Xva = torch.from_numpy(X[va]); yva = y[va]

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))   # 14
        self.conv2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.ReLU())                   # 14 (마지막 conv)
        self.pool = nn.MaxPool2d(2)                                                            # 7
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10))
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)         # <- Grad-CAM 대상 활성
        return self.head(self.pool(x))

model = CNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
for epoch in range(3):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
@torch.no_grad()
def acc():
    model.eval(); p=[]
    for i in range(0,len(Xva),512): p.append(model(Xva[i:i+512].to(device)).argmax(1).cpu().numpy())
    return (np.concatenate(p)==yva).mean()
print("CNN val acc:", round(float(acc()),4))

## 3. Grad-CAM

마지막 conv 활성과 그 그래디언트를 후킹해, 채널별 평균 그래디언트로 가중합 → ReLU → 입력 크기로 확대.

In [ ]:
import torch.nn.functional as F

_feat = {}
def fwd_hook(m, i, o): _feat["act"] = o
def bwd_hook(m, gi, go): _feat["grad"] = go[0]
h1 = model.conv2.register_forward_hook(fwd_hook)
h2 = model.conv2.register_full_backward_hook(bwd_hook)

def grad_cam(x):  # x: (1,1,28,28)
    model.eval()
    logit = model(x.to(device)); cls = logit.argmax(1)
    model.zero_grad(); logit[0, cls].backward()
    act, grad = _feat["act"][0], _feat["grad"][0]      # (C,H,W)
    weights = grad.mean(dim=(1,2))                      # (C,)
    cam = F.relu((weights[:, None, None] * act).sum(0)) # (H,W)
    cam = cam / (cam.max() + 1e-8)
    cam = F.interpolate(cam[None,None], size=(28,28), mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy(), int(cls)

cam, pred = grad_cam(Xva[0:1])
print("Grad-CAM shape:", cam.shape, "| pred:", pred)

## 4. Saliency (입력 그래디언트)

예측 클래스 점수의 입력 픽셀에 대한 그래디언트 절댓값 = 픽셀 중요도.

In [ ]:
def saliency(x):
    model.eval(); x = x.clone().to(device).requires_grad_(True)
    logit = model(x); cls = logit.argmax(1)
    model.zero_grad(); logit[0, cls].backward()
    sal = x.grad.abs()[0,0].cpu().numpy()
    return sal / (sal.max() + 1e-8)

sal = saliency(Xva[0:1])
print("Saliency shape:", sal.shape)

## 5. 시각화

원본 / Grad-CAM 오버레이 / Saliency 를 나란히 표시.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for k in range(4):
    img = Xva[k:k+1]
    cam, pred = grad_cam(img); sal = saliency(img)
    base = img[0,0].numpy()
    ax[0,k].imshow(base, cmap="gray"); ax[0,k].set_title(f"input (pred {pred})")
    ax[1,k].imshow(base, cmap="gray"); ax[1,k].imshow(cam, cmap="jet", alpha=0.5); ax[1,k].set_title("Grad-CAM")
    ax[2,k].imshow(sal, cmap="hot"); ax[2,k].set_title("Saliency")
    for r in range(3): ax[r,k].axis("off")
plt.tight_layout(); plt.show()
h1.remove(); h2.remove()

## 🏁 제출 안내

이 노트북은 **모델 해석(중요 픽셀 시각화)** 데모입니다(제출 리더보드 없음).

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**

> IOAI 2025 *Pixel Efficiency*(가장 중요한 픽셀 식별) 과제의 기반 기법입니다. 공식 해법은 CLIP 비전 모델의 기여도를 활용합니다.